# EduSearch / Education IR Lab
Run the cells from top to bottom. The first cell uploads the project ZIP. The build uses Spark, not the fallback. No credentials are needed.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload exactly one project ZIP"
project = Path("/content/edusearch_project")
project.mkdir(exist_ok=True)
with ZipFile(archives[0]) as z:
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in z.namelist())
    z.extractall(project)
os.chdir(project)
assert Path("src/build.py").exists(), "Upload the full project ZIP, not the dataset-only ZIP"

## 1. Install and build with Spark
Java 17 or 21 is required. The build exports all matrices from the Spark positional index.

In [ ]:
!java -version
%pip install -q -r requirements.txt
import subprocess, sys
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
subprocess.run([sys.executable, "src/build.py", "--engine", "spark"], check=True)

## 2. Verify and inspect results

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
import pandas as pd
from IPython.display import display
for name in ["positional_index", "tf", "weighted_tf", "df_idf", "tfidf", "document_lengths", "normalized_tfidf", "rankings"]:
    print(name)
    display(pd.read_csv(f"outputs/{name}.csv"))

## 3. Download regenerated project
The new ZIP contains Spark-generated data and outputs. The included report describes the original independent reference validation; add screenshots of your actual Spark run for submission.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive = Path("/content/edusearch-spark.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in project.rglob("*"):
        if p.is_file() and not any(x in {"__pycache__", ".git", "spark-warehouse", ".venv"} for x in p.relative_to(project).parts):
            z.write(p, p.relative_to(project))
files.download(str(archive))